In [59]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque

In [60]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [61]:
df = pd.read_csv('data/results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49407 non-null  float64
 4   away_score  49407 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 5.9 MB


In [63]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

In [64]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [65]:
df = df[df['date'] >= '2000-01-01']
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24062,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False
24063,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False
24064,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False
24068,2000-01-09,Mexico,Iran,2.0,1.0,Friendly,Oakland,United States,True
24067,2000-01-09,Ivory Coast,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False


In [66]:
test = df[df['date'] >= '2026-06-11']

In [67]:
test.to_csv("data/test_data.csv", index=False)

In [68]:
test.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49405,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup,Mexico City,Mexico,False
49407,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True


In [69]:
df = df[df['date'] < '2026-06-11']

In [70]:
df.tournament.value_counts().sort_index()

tournament
ABCS Tournament                                               20
AFC Asian Cup                                                256
AFC Asian Cup qualification                                  530
AFC Challenge Cup                                             87
AFC Challenge Cup qualification                               92
AFC Solidarity Cup                                            13
AFF Championship                                             251
AFF Championship qualification                                56
ASEAN Championship                                            26
ASEAN Championship qualification                               2
African Cup of Nations                                       525
African Cup of Nations qualification                        1416
Afro-Asian Games                                               4
Al Ain International Cup                                       4
Amílcar Cabral Cup                                            30
Arab Cup      

In [71]:
def classify_tournament(tournament):
    t = str(tournament).lower().strip()

    # 1. Amistosos
    if t == "friendly":
        return "Friendly", 1

    # 6. Mundial
    if t == "fifa world cup":
        return "World Cup", 6

    # 4. Nations League
    if "nations league" in t and "qualification" not in t:
        return "Nations League", 4

    # 3. Clasificatorios
    if (
        "qualification" in t
        or "qualifier" in t
        or "qualifying" in t
    ):
        return "Qualification", 3

    # 5. Copas continentales principales
    continental_cups = [
        "uefa euro",
        "copa américa",
        "copa america",
        "african cup of nations",
        "afc asian cup",
        "gold cup",
        "oceania nations cup",
        "confederations cup"
    ]

    if t in continental_cups:
        return "Continental Championship", 5

    # 2. Torneos regionales / menores
    return "Regional / Minor Tournament", 2

In [72]:
df[["competition_category", "competition_level"]] = df["tournament"].apply(
    lambda x: pd.Series(classify_tournament(x))
)

In [73]:
df[["tournament", "competition_category", "competition_level"]].drop_duplicates()

,tournament,competition_category,competition_level
24062,Friendly,Friendly,1
24082,African Cup of Nations,Continental Championship,5
24083,AFC Asian Cup qualification,Qualification,3
24116,Nordic Championship,Regional / Minor Tournament,2
24126,Cyprus International Tournament,Regional / Minor Tournament,2
24138,Lunar New Year Cup,Regional / Minor Tournament,2
24146,Malta International Tournament,Regional / Minor Tournament,2
24167,Gold Cup,Continental Championship,5
24212,King's Cup,Regional / Minor Tournament,2
24246,FIFA World Cup qualification,Qualification,3


In [74]:
df["competition_category"].value_counts()

competition_category
Qualification                  9974
Friendly                       8448
Regional / Minor Tournament    3597
Continental Championship       1860
Nations League                 1080
World Cup                       384
Name: count, dtype: int64

In [75]:
df = df.drop(
    columns=[
        "tournament",
        "city",
        "country",
        "competition_category"
    ]
)

In [76]:
df["neutral"] = df["neutral"].astype(int)

In [77]:
df.head()

,date,home_team,away_team,home_score,away_score,neutral,competition_level
24062,2000-01-04,Egypt,Togo,2.0,1.0,0,1
24063,2000-01-07,Tunisia,Togo,7.0,0.0,0,1
24064,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,1
24068,2000-01-09,Mexico,Iran,2.0,1.0,1,1
24067,2000-01-09,Ivory Coast,Egypt,2.0,0.0,0,1


In [78]:
df = df.sort_values("date").reset_index(drop=True)

In [79]:
df.head()

,date,home_team,away_team,home_score,away_score,neutral,competition_level
0,2000-01-04,Egypt,Togo,2.0,1.0,0,1
1,2000-01-07,Tunisia,Togo,7.0,0.0,0,1
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,1
3,2000-01-09,Mexico,Iran,2.0,1.0,1,1
4,2000-01-09,Ivory Coast,Egypt,2.0,0.0,0,1


In [80]:
def get_result(gf, ga):
    if gf > ga:
        return "W"
    elif gf == ga:
        return "D"
    else:
        return "L"


def calculate_history_features(history, window):
    if len(history) == 0:
        return {
            f"GF{window}": np.nan,
            f"GA{window}": np.nan,
            f"W{window}": 0,
            f"D{window}": 0,
            f"L{window}": 0,
            f"Points{window}": 0,
            f"Mean_competition{window}": np.nan,
            f"Matches_used{window}": 0
        }

    goals_for = [m["goals_for"] for m in history]
    goals_against = [m["goals_against"] for m in history]
    competitions = [m["competition_level"] for m in history]
    results = [m["result"] for m in history]

    wins = results.count("W")
    draws = results.count("D")
    losses = results.count("L")

    return {
        f"GF{window}": np.mean(goals_for),
        f"GA{window}": np.mean(goals_against),
        f"W{window}": wins,
        f"D{window}": draws,
        f"L{window}": losses,
        f"Points{window}": wins * 3 + draws,
        f"Mean_competition{window}": np.mean(competitions),
        f"Matches_used{window}": len(history)
    }


def calculate_streak(history, streak_type="win"):
    streak = 0

    for match in reversed(history):
        result = match["result"]

        if streak_type == "win":
            if result == "W":
                streak += 1
            else:
                break

        elif streak_type == "unbeaten":
            if result in ["W", "D"]:
                streak += 1
            else:
                break

    return streak


def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(home_elo, away_elo, home_score, away_score, competition_level):
    if home_score > away_score:
        actual_home = 1
        actual_away = 0
    elif home_score == away_score:
        actual_home = 0.5
        actual_away = 0.5
    else:
        actual_home = 0
        actual_away = 1

    k_values = {
        1: 10,  # Friendly
        2: 15,  # Regional / minor
        3: 20,  # Qualification
        4: 25,  # Nations League
        5: 35,  # Continental Championship
        6: 50   # World Cup
    }

    k = k_values.get(competition_level, 20)

    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)

    new_home_elo = home_elo + k * (actual_home - expected_home)
    new_away_elo = away_elo + k * (actual_away - expected_away)

    return new_home_elo, new_away_elo

In [81]:
def get_result(gf, ga):
    if gf > ga:
        return "W"
    elif gf == ga:
        return "D"
    else:
        return "L"


def calculate_history_features(history, window):
    if len(history) == 0:
        return {
            f"GF{window}": np.nan,
            f"GA{window}": np.nan,
            f"W{window}": 0,
            f"D{window}": 0,
            f"L{window}": 0,
            f"Points{window}": 0,
            f"Mean_competition{window}": np.nan,
            f"Matches_used{window}": 0
        }

    goals_for = [m["goals_for"] for m in history]
    goals_against = [m["goals_against"] for m in history]
    competitions = [m["competition_level"] for m in history]
    results = [m["result"] for m in history]

    wins = results.count("W")
    draws = results.count("D")
    losses = results.count("L")

    return {
        f"GF{window}": np.mean(goals_for),
        f"GA{window}": np.mean(goals_against),
        f"W{window}": wins,
        f"D{window}": draws,
        f"L{window}": losses,
        f"Points{window}": wins * 3 + draws,
        f"Mean_competition{window}": np.mean(competitions),
        f"Matches_used{window}": len(history)
    }


def calculate_streak(history, streak_type="win"):
    streak = 0

    for match in reversed(history):
        result = match["result"]

        if streak_type == "win":
            if result == "W":
                streak += 1
            else:
                break

        elif streak_type == "unbeaten":
            if result in ["W", "D"]:
                streak += 1
            else:
                break

    return streak


def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(home_elo, away_elo, home_score, away_score, competition_level):
    if home_score > away_score:
        actual_home = 1
        actual_away = 0
    elif home_score == away_score:
        actual_home = 0.5
        actual_away = 0.5
    else:
        actual_home = 0
        actual_away = 1

    k_values = {
        1: 10,  # Friendly
        2: 15,  # Regional / minor
        3: 20,  # Qualification
        4: 25,  # Nations League
        5: 35,  # Continental Championship
        6: 50   # World Cup
    }

    k = k_values.get(competition_level, 20)

    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)

    new_home_elo = home_elo + k * (actual_home - expected_home)
    new_away_elo = away_elo + k * (actual_away - expected_away)

    return new_home_elo, new_away_elo

In [82]:
team_history5 = defaultdict(lambda: deque(maxlen=5))
team_history10 = defaultdict(lambda: deque(maxlen=10))

elo = defaultdict(lambda: 1500)

h2h_history = defaultdict(list)

rows = []

for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    competition_level = match["competition_level"]

    # =========================
    # Historial antes del partido
    # =========================

    home_hist5 = team_history5[home_team]
    away_hist5 = team_history5[away_team]

    home_hist10 = team_history10[home_team]
    away_hist10 = team_history10[away_team]

    home_f5 = calculate_history_features(home_hist5, 5)
    away_f5 = calculate_history_features(away_hist5, 5)

    home_f10 = calculate_history_features(home_hist10, 10)
    away_f10 = calculate_history_features(away_hist10, 10)

    # =========================
    # Rachas antes del partido
    # =========================

    home_win_streak = calculate_streak(home_hist10, "win")
    away_win_streak = calculate_streak(away_hist10, "win")

    home_unbeaten_streak = calculate_streak(home_hist10, "unbeaten")
    away_unbeaten_streak = calculate_streak(away_hist10, "unbeaten")

    # =========================
    # Elo antes del partido
    # =========================

    home_elo = elo[home_team]
    away_elo = elo[away_team]
    elo_diff = home_elo - away_elo

    # =========================
    # H2H antes del partido
    # =========================

    h2h_key = tuple(sorted([home_team, away_team]))
    previous_h2h = h2h_history[h2h_key]

    h2h_home_wins = sum(
        1 for m in previous_h2h
        if m["winner"] == home_team
    )

    h2h_away_wins = sum(
        1 for m in previous_h2h
        if m["winner"] == away_team
    )

    h2h_draws = sum(
        1 for m in previous_h2h
        if m["winner"] == "Draw"
    )

    h2h_matches = len(previous_h2h)

    # =========================
    # Crear fila del modelo
    # =========================

    row = {
        "date": match["date"],
        "home_team": home_team,
        "away_team": away_team,

        # Últimos 5 local
        "home_GF5": home_f5["GF5"],
        "home_GA5": home_f5["GA5"],
        "home_W5": home_f5["W5"],
        "home_D5": home_f5["D5"],
        "home_L5": home_f5["L5"],
        "home_points5": home_f5["Points5"],
        "home_mean_competition5": home_f5["Mean_competition5"],
        "home_matches_used5": home_f5["Matches_used5"],

        # Últimos 5 visitante
        "away_GF5": away_f5["GF5"],
        "away_GA5": away_f5["GA5"],
        "away_W5": away_f5["W5"],
        "away_D5": away_f5["D5"],
        "away_L5": away_f5["L5"],
        "away_points5": away_f5["Points5"],
        "away_mean_competition5": away_f5["Mean_competition5"],
        "away_matches_used5": away_f5["Matches_used5"],

        # Últimos 10 local
        "home_GF10": home_f10["GF10"],
        "home_GA10": home_f10["GA10"],
        "home_points10": home_f10["Points10"],
        "home_mean_competition10": home_f10["Mean_competition10"],
        "home_matches_used10": home_f10["Matches_used10"],

        # Últimos 10 visitante
        "away_GF10": away_f10["GF10"],
        "away_GA10": away_f10["GA10"],
        "away_points10": away_f10["Points10"],
        "away_mean_competition10": away_f10["Mean_competition10"],
        "away_matches_used10": away_f10["Matches_used10"],

        # Elo
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_diff": elo_diff,

        # Rachas
        "home_win_streak": home_win_streak,
        "away_win_streak": away_win_streak,
        "home_unbeaten_streak": home_unbeaten_streak,
        "away_unbeaten_streak": away_unbeaten_streak,

        # Historial directo
        "h2h_home_wins": h2h_home_wins,
        "h2h_away_wins": h2h_away_wins,
        "h2h_draws": h2h_draws,
        "h2h_matches": h2h_matches,

        # Contexto del partido
        "neutral": match["neutral"],
        "competition_level": competition_level,

        # Targets
        "home_score": home_score,
        "away_score": away_score
    }

    rows.append(row)

    # =========================
    # Actualizar historiales DESPUÉS del partido
    # =========================

    home_result = get_result(home_score, away_score)
    away_result = get_result(away_score, home_score)

    home_match_record = {
        "goals_for": home_score,
        "goals_against": away_score,
        "competition_level": competition_level,
        "result": home_result
    }

    away_match_record = {
        "goals_for": away_score,
        "goals_against": home_score,
        "competition_level": competition_level,
        "result": away_result
    }

    team_history5[home_team].append(home_match_record)
    team_history5[away_team].append(away_match_record)

    team_history10[home_team].append(home_match_record)
    team_history10[away_team].append(away_match_record)

    # Actualizar Elo
    new_home_elo, new_away_elo = update_elo(
        home_elo,
        away_elo,
        home_score,
        away_score,
        competition_level
    )

    elo[home_team] = new_home_elo
    elo[away_team] = new_away_elo

    # Actualizar H2H
    if home_score > away_score:
        winner = home_team
    elif home_score < away_score:
        winner = away_team
    else:
        winner = "Draw"

    h2h_history[h2h_key].append({
        "home_team": home_team,
        "away_team": away_team,
        "winner": winner,
        "home_score": home_score,
        "away_score": away_score
    })

features_df = pd.DataFrame(rows)

In [83]:
features_df.head()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,home_matches_used5,away_GF5,away_GA5,away_W5,away_D5,away_L5,away_points5,away_mean_competition5,away_matches_used5,home_GF10,home_GA10,home_points10,home_mean_competition10,home_matches_used10,away_GF10,away_GA10,away_points10,away_mean_competition10,away_matches_used10,home_elo,away_elo,elo_diff,home_win_streak,away_win_streak,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
0,2000-01-04,Egypt,Togo,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,NaN,0,NaN,NaN,0,NaN,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,0,1,2.0,1.0
1,2000-01-07,Tunisia,Togo,NaN,NaN,0,0,0,0,NaN,0,1.0,2.0,0,0,1,0,1.0,1,NaN,NaN,0,NaN,0,1.0,2.0,0,1.0,1,1500.0,1495.0,5.0,0,0,0,0,0,0,0,0,0,1,7.0,0.0
2,2000-01-08,Trinidad and Tobago,Canada,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,NaN,0,NaN,NaN,0,NaN,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,0,1,0.0,0.0
3,2000-01-09,Mexico,Iran,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,0,0,0,NaN,0,NaN,NaN,0,NaN,0,NaN,NaN,0,NaN,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,1,1,2.0,1.0
4,2000-01-09,Ivory Coast,Egypt,NaN,NaN,0,0,0,0,NaN,0,2.0,1.0,1,0,0,3,1.0,1,NaN,NaN,0,NaN,0,2.0,1.0,3,1.0,1,1500.0,1505.0,-5.0,0,1,0,1,0,0,0,0,0,1,2.0,0.0


In [84]:
features_df = features_df.fillna(0)
features_df.head()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,home_matches_used5,away_GF5,away_GA5,away_W5,away_D5,away_L5,away_points5,away_mean_competition5,away_matches_used5,home_GF10,home_GA10,home_points10,home_mean_competition10,home_matches_used10,away_GF10,away_GA10,away_points10,away_mean_competition10,away_matches_used10,home_elo,away_elo,elo_diff,home_win_streak,away_win_streak,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
0,2000-01-04,Egypt,Togo,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,0,1,2.0,1.0
1,2000-01-07,Tunisia,Togo,0.0,0.0,0,0,0,0,0.0,0,1.0,2.0,0,0,1,0,1.0,1,0.0,0.0,0,0.0,0,1.0,2.0,0,1.0,1,1500.0,1495.0,5.0,0,0,0,0,0,0,0,0,0,1,7.0,0.0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,0,1,0.0,0.0
3,2000-01-09,Mexico,Iran,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,1500.0,1500.0,0.0,0,0,0,0,0,0,0,0,1,1,2.0,1.0
4,2000-01-09,Ivory Coast,Egypt,0.0,0.0,0,0,0,0,0.0,0,2.0,1.0,1,0,0,3,1.0,1,0.0,0.0,0,0.0,0,2.0,1.0,3,1.0,1,1500.0,1505.0,-5.0,0,1,0,1,0,0,0,0,0,1,2.0,0.0


In [85]:
features_df.tail()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,home_matches_used5,away_GF5,away_GA5,away_W5,away_D5,away_L5,away_points5,away_mean_competition5,away_matches_used5,home_GF10,home_GA10,home_points10,home_mean_competition10,home_matches_used10,away_GF10,away_GA10,away_points10,away_mean_competition10,away_matches_used10,home_elo,away_elo,elo_diff,home_win_streak,away_win_streak,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
25338,2026-06-09,Liberia,Sierra Leone,0.6,1.4,0,2,3,2,1.8,5,1.2,0.6,3,1,1,10,2.4,5,1.0,1.5,9,2.4,10,1.1,0.9,15,2.7,10,1414.715497,1457.315001,-42.599504,0,1,0,3,4,4,1,9,0,1,3.0,1.0
25339,2026-06-10,Portugal,Nigeria,2.6,0.8,3,1,1,10,1.8,5,2.2,1.0,3,2,0,11,1.4,5,2.6,1.0,21,2.5,10,2.0,0.6,22,3.2,10,1839.739806,1745.237357,94.502450,2,0,4,10,1,0,0,1,0,1,2.0,1.0
25340,2026-06-10,Bolivia,Algeria,1.2,1.6,2,0,3,6,1.8,5,1.8,0.4,3,1,1,10,2.6,5,0.8,1.7,10,1.5,10,1.9,0.4,23,3.2,10,1528.265175,1744.511065,-216.245890,0,1,0,3,0,1,0,1,1,1,0.0,4.0
25341,2026-06-10,England,Costa Rica,1.2,0.4,3,1,1,10,1.8,5,0.6,2.2,0,2,3,2,1.8,5,2.2,0.5,22,2.0,10,1.3,1.8,9,2.6,10,1856.466253,1669.175765,187.290488,1,0,1,0,1,0,1,2,1,1,3.0,0.0
25342,2026-06-10,Afghanistan,Pakistan,0.4,0.8,1,2,2,5,2.6,5,1.2,1.2,2,2,1,8,2.6,5,0.4,1.4,5,2.2,10,0.6,2.2,8,2.8,10,1364.177601,1274.055836,90.121765,0,2,0,2,1,4,2,7,1,2,0.0,2.0


In [86]:
features_df.to_csv("data/features.csv", index=False)